# BCH Alpha Hunter Engine — GitHub Automation Ready

Daily alpha scanner for bull call spread opportunities, breakout continuation, IV condition, gamma acceleration proxy, and earnings drift placeholder.

**Automation mode:** This notebook is prepared to run unattended in GitHub Actions. It pulls market/options data through `yfinance`, creates stable report files, and uploads them through the GitHub workflow artifact step.

**Important:** Research/decision-support only. Verify live option chain, Greeks, liquidity, event risk, and broker execution quality before approval.

In [ ]:
# =========================
# Environment / Dependencies
# GitHub-ready and Colab-ready
# =========================

import sys
import subprocess
import importlib.util

REQUIRED_PACKAGES = ["yfinance", "pandas", "numpy", "matplotlib", "scipy", "openpyxl"]

missing = [pkg for pkg in REQUIRED_PACKAGES if importlib.util.find_spec(pkg) is None]
if missing:
    print("Installing missing packages:", missing)
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", *missing])
else:
    print("All required packages already installed.")

In [ ]:
# =========================
# Imports and Runtime Setup
# =========================

import os
import json
import warnings
warnings.filterwarnings("ignore")

import math
import numpy as np
import pandas as pd
import yfinance as yf
import matplotlib.pyplot as plt

from pathlib import Path
from scipy.stats import norm
from datetime import datetime, timedelta, timezone

pd.set_option("display.max_columns", 200)
pd.set_option("display.width", 200)

RUN_TS = datetime.now(timezone.utc).strftime("%Y-%m-%dT%H:%M:%SZ")
RUN_ID = datetime.now(timezone.utc).strftime("%Y%m%d_%H%M%S")
IS_GITHUB_ACTIONS = os.environ.get("GITHUB_ACTIONS", "false").lower() == "true"

OUTPUT_DIR = Path(os.environ.get("BCH_OUTPUT_DIR", ".")).resolve()
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

print("BCH Alpha Hunter Engine loaded")
print("Run timestamp UTC:", RUN_TS)
print("GitHub Actions mode:", IS_GITHUB_ACTIONS)
print("Output directory:", OUTPUT_DIR)

In [ ]:
# =========================
# BCH Configuration
# =========================

# Override in GitHub Actions with env variable BCH_UNIVERSE="NVDA,AMD,MSFT"
DEFAULT_UNIVERSE = ["NVDA", "MSFT", "AMZN", "META", "GOOGL", "AMD", "AVGO", "TSM", "AAPL", "TSLA"]
BCH_UNIVERSE = [x.strip().upper() for x in os.environ.get("BCH_UNIVERSE", ",".join(DEFAULT_UNIVERSE)).split(",") if x.strip()]

MIN_DTE = int(os.environ.get("MIN_DTE", "30"))
MAX_DTE = int(os.environ.get("MAX_DTE", "90"))

TARGET_ROI = float(os.environ.get("TARGET_ROI", "0.20"))
SECOND_TARGET_ROI = float(os.environ.get("SECOND_TARGET_ROI", "0.36"))
MIN_PROB_TARGET = float(os.environ.get("MIN_PROB_TARGET", "0.70"))

MAX_SPREAD_WIDTH = float(os.environ.get("MAX_SPREAD_WIDTH", "30"))
MIN_SPREAD_WIDTH = float(os.environ.get("MIN_SPREAD_WIDTH", "5"))

RISK_FREE_RATE = float(os.environ.get("RISK_FREE_RATE", "0.045"))
HIST_PERIOD = os.environ.get("HIST_PERIOD", "1y")
HIST_INTERVAL = os.environ.get("HIST_INTERVAL", "1d")

print("BCH Universe:", BCH_UNIVERSE)
print("DTE Window:", MIN_DTE, "to", MAX_DTE)
print("Target ROI:", TARGET_ROI, "Second Target ROI:", SECOND_TARGET_ROI)

In [ ]:
# =========================
# Helper Functions
# =========================

def safe_float(x):
    try:
        if pd.isna(x):
            return np.nan
        return float(x)
    except Exception:
        return np.nan

def get_history(ticker, period=HIST_PERIOD, interval=HIST_INTERVAL):
    df = yf.download(ticker, period=period, interval=interval, auto_adjust=True, progress=False)
    if df.empty:
        return df
    if isinstance(df.columns, pd.MultiIndex):
        df.columns = [c[0] for c in df.columns]
    df = df.dropna()
    return df

def rsi(series, period=14):
    delta = series.diff()
    gain = delta.clip(lower=0).rolling(period).mean()
    loss = -delta.clip(upper=0).rolling(period).mean()
    rs = gain / loss.replace(0, np.nan)
    return 100 - (100 / (1 + rs))

def add_technical_features(df):
    out = df.copy()
    out["ret_1d"] = out["Close"].pct_change()
    out["ret_5d"] = out["Close"].pct_change(5)
    out["ret_20d"] = out["Close"].pct_change(20)
    out["sma_20"] = out["Close"].rolling(20).mean()
    out["sma_50"] = out["Close"].rolling(50).mean()
    out["sma_200"] = out["Close"].rolling(200).mean()
    out["rsi_14"] = rsi(out["Close"], 14)
    out["vol_20"] = out["ret_1d"].rolling(20).std() * np.sqrt(252)
    out["vol_60"] = out["ret_1d"].rolling(60).std() * np.sqrt(252)
    out["volume_sma_20"] = out["Volume"].rolling(20).mean()
    out["volume_ratio"] = out["Volume"] / out["volume_sma_20"]
    out["high_20"] = out["High"].rolling(20).max()
    out["breakout_20"] = out["Close"] >= out["high_20"].shift(1)
    out["trend_slope_20"] = out["Close"].rolling(20).apply(lambda x: np.polyfit(range(len(x)), x, 1)[0] if len(x) == 20 else np.nan)
    return out

def black_scholes_call(S, K, T, r, sigma):
    if T <= 0 or sigma <= 0 or S <= 0 or K <= 0:
        return np.nan
    d1 = (np.log(S/K) + (r + 0.5*sigma*sigma)*T) / (sigma*np.sqrt(T))
    d2 = d1 - sigma*np.sqrt(T)
    return S*norm.cdf(d1) - K*np.exp(-r*T)*norm.cdf(d2)

def prob_touch_proxy(S, target_price, T, sigma):
    # Simplified lognormal probability of finishing above target by expiration.
    # Touch probability is often higher than finish probability, so this is conservative-ish for early target analysis.
    if S <= 0 or target_price <= 0 or T <= 0 or sigma <= 0:
        return np.nan
    z = (np.log(target_price/S) - (-0.5*sigma*sigma)*T) / (sigma*np.sqrt(T))
    return 1 - norm.cdf(z)

def nearest_expirations(ticker_obj, min_dte=MIN_DTE, max_dte=MAX_DTE):
    today = datetime.utcnow().date()
    good = []
    for exp in ticker_obj.options:
        d = datetime.strptime(exp, "%Y-%m-%d").date()
        dte = (d - today).days
        if min_dte <= dte <= max_dte:
            good.append((exp, dte))
    return good

def score_breakout(features):
    score = 0
    reasons = []
    if bool(features.get("breakout_20", False)):
        score += 25; reasons.append("20-day breakout")
    if features.get("Close", 0) > features.get("sma_20", np.inf):
        score += 15; reasons.append("above SMA20")
    if features.get("sma_20", 0) > features.get("sma_50", np.inf):
        score += 15; reasons.append("SMA20 > SMA50")
    if features.get("rsi_14", 0) >= 55 and features.get("rsi_14", 0) <= 75:
        score += 15; reasons.append("RSI momentum zone")
    if features.get("volume_ratio", 0) >= 1.2:
        score += 15; reasons.append("volume expansion")
    if features.get("trend_slope_20", 0) > 0:
        score += 15; reasons.append("positive trend slope")
    return min(score, 100), "; ".join(reasons)

def score_iv_mispricing(hist_vol, implied_vol):
    if pd.isna(hist_vol) or pd.isna(implied_vol) or implied_vol <= 0:
        return 0, "insufficient IV/HV data"
    iv_hv_ratio = implied_vol / hist_vol if hist_vol and hist_vol > 0 else np.nan

    # For long debit spreads, prefer IV not massively overpriced.
    if iv_hv_ratio < 0.85:
        return 90, f"IV below historical vol; IV/HV={iv_hv_ratio:.2f}"
    elif iv_hv_ratio < 1.05:
        return 70, f"IV near fair value; IV/HV={iv_hv_ratio:.2f}"
    elif iv_hv_ratio < 1.25:
        return 45, f"IV moderately rich; IV/HV={iv_hv_ratio:.2f}"
    else:
        return 20, f"IV expensive; IV/HV={iv_hv_ratio:.2f}"

def score_gamma_acceleration(delta_long, gamma_long, dte):
    if pd.isna(delta_long) or pd.isna(gamma_long):
        return 0, "missing greeks"
    # Favor positive gamma with useful delta.
    raw = (delta_long * 60) + (gamma_long * 2000) + max(0, 60-dte) * 0.25
    score = max(0, min(100, raw))
    return score, f"delta={delta_long:.2f}, gamma={gamma_long:.4f}, dte={dte}"

def score_earnings_drift(ticker):
    # yfinance earnings dates can be inconsistent; this is a placeholder scoring framework.
    # User can replace this with Nasdaq/Polygon/Finnhub earnings API.
    return 50, "neutral placeholder; connect confirmed earnings calendar API for full drift scoring"

In [ ]:
# =========================
# Bull Call Spread Scanner
# =========================

def scan_bull_call_spreads(ticker):
    tk = yf.Ticker(ticker)
    hist = get_history(ticker)
    if hist.empty or len(hist) < 60:
        return pd.DataFrame()

    feat = add_technical_features(hist).iloc[-1].to_dict()
    S = safe_float(feat["Close"])
    hist_vol = safe_float(feat.get("vol_60", np.nan))

    breakout_score, breakout_reason = score_breakout(feat)
    earnings_score, earnings_reason = score_earnings_drift(ticker)

    rows = []
    try:
        expirations = nearest_expirations(tk)
    except Exception as e:
        print(f"{ticker}: options unavailable:", e)
        return pd.DataFrame()

    for exp, dte in expirations[:8]:
        try:
            chain = tk.option_chain(exp)
            calls = chain.calls.copy()
        except Exception:
            continue

        if calls.empty:
            continue

        calls["strike"] = calls["strike"].astype(float)
        calls["mid"] = (calls["bid"].fillna(0) + calls["ask"].fillna(0)) / 2
        calls["mid"] = calls["mid"].where(calls["mid"] > 0, calls["lastPrice"])
        calls = calls[(calls["strike"] >= S*0.90) & (calls["strike"] <= S*1.20)].copy()
        calls = calls.dropna(subset=["strike", "mid"])

        for _, long in calls.iterrows():
            K1 = safe_float(long["strike"])
            long_mid = safe_float(long["mid"])
            if pd.isna(long_mid) or long_mid <= 0:
                continue

            upper_calls = calls[(calls["strike"] > K1 + MIN_SPREAD_WIDTH - 0.01) & (calls["strike"] <= K1 + MAX_SPREAD_WIDTH + 0.01)]
            for _, short in upper_calls.iterrows():
                K2 = safe_float(short["strike"])
                short_mid = safe_float(short["mid"])
                if pd.isna(short_mid) or short_mid <= 0:
                    continue

                width = K2 - K1
                debit = long_mid - short_mid
                if debit <= 0 or debit >= width:
                    continue

                max_profit = width - debit
                max_roi = max_profit / debit

                target_value_20 = debit * (1 + TARGET_ROI)
                target_value_36 = debit * (1 + SECOND_TARGET_ROI)

                # Estimate underlying price needed for target spread value.
                # Conservative linear intrinsic approximation.
                target_underlying_20 = K1 + target_value_20
                target_underlying_36 = K1 + target_value_36

                iv_long = safe_float(long.get("impliedVolatility", np.nan))
                iv_short = safe_float(short.get("impliedVolatility", np.nan))
                spread_iv = np.nanmean([iv_long, iv_short])
                sigma = spread_iv if not pd.isna(spread_iv) and spread_iv > 0 else hist_vol
                T = dte / 365

                prob_20 = prob_touch_proxy(S, target_underlying_20, T, sigma)
                prob_36 = prob_touch_proxy(S, target_underlying_36, T, sigma)

                iv_score, iv_reason = score_iv_mispricing(hist_vol, spread_iv)

                delta_long = safe_float(long.get("delta", np.nan)) if "delta" in long else np.nan
                gamma_long = safe_float(long.get("gamma", np.nan)) if "gamma" in long else np.nan
                gamma_score, gamma_reason = score_gamma_acceleration(delta_long, gamma_long, dte)

                # Some free data sources do not include greeks. Use fallback proxy.
                if gamma_score == 0:
                    moneyness = S / K1
                    gamma_score = max(0, min(100, 100 - abs(moneyness - 1) * 300))
                    gamma_reason = f"fallback moneyness gamma proxy; S/K1={moneyness:.2f}"

                liquidity_score = 100
                long_spread = safe_float(long.get("ask", np.nan)) - safe_float(long.get("bid", np.nan))
                short_spread = safe_float(short.get("ask", np.nan)) - safe_float(short.get("bid", np.nan))
                if not pd.isna(long_spread) and not pd.isna(short_spread):
                    avg_option_spread = np.nanmean([long_spread/max(long_mid, .01), short_spread/max(short_mid, .01)])
                    liquidity_score = max(0, min(100, 100 - avg_option_spread*250))

                bch_score = (
                    0.35 * (prob_20 * 100 if not pd.isna(prob_20) else 0) +
                    0.20 * breakout_score +
                    0.15 * iv_score +
                    0.15 * gamma_score +
                    0.10 * liquidity_score +
                    0.05 * earnings_score
                )

                decision = "WATCH"
                if prob_20 >= MIN_PROB_TARGET and bch_score >= 75:
                    decision = "BCH QUALIFIED"
                elif bch_score >= 65:
                    decision = "RESEARCH"

                rows.append({
                    "ticker": ticker,
                    "spot": round(S, 2),
                    "expiration": exp,
                    "dte": dte,
                    "long_call": K1,
                    "short_call": K2,
                    "width": width,
                    "debit_mid": round(debit, 2),
                    "max_profit": round(max_profit, 2),
                    "max_roi_%": round(max_roi*100, 1),
                    "target_20_value": round(target_value_20, 2),
                    "target_36_value": round(target_value_36, 2),
                    "underlying_needed_20_proxy": round(target_underlying_20, 2),
                    "underlying_needed_36_proxy": round(target_underlying_36, 2),
                    "prob_20_proxy_%": round(prob_20*100, 1) if not pd.isna(prob_20) else np.nan,
                    "prob_36_proxy_%": round(prob_36*100, 1) if not pd.isna(prob_36) else np.nan,
                    "breakout_score": round(breakout_score, 1),
                    "iv_score": round(iv_score, 1),
                    "gamma_score": round(gamma_score, 1),
                    "liquidity_score": round(liquidity_score, 1),
                    "earnings_score": round(earnings_score, 1),
                    "bch_score": round(bch_score, 1),
                    "decision": decision,
                    "breakout_reason": breakout_reason,
                    "iv_reason": iv_reason,
                    "gamma_reason": gamma_reason,
                    "earnings_reason": earnings_reason
                })
    return pd.DataFrame(rows)

In [ ]:
# =========================
# Run BCH Alpha Hunter Scanner
# =========================

run_log = []
all_results = []

for ticker in BCH_UNIVERSE:
    msg = f"Scanning {ticker}..."
    print(msg)
    run_log.append({"ticker": ticker, "status": "started", "message": msg})
    try:
        df = scan_bull_call_spreads(ticker)
        if not df.empty:
            all_results.append(df)
            run_log.append({"ticker": ticker, "status": "ok", "rows": int(len(df))})
            print(f"{ticker}: {len(df)} candidate rows")
        else:
            run_log.append({"ticker": ticker, "status": "empty", "rows": 0})
            print(f"{ticker}: no candidate rows")
    except Exception as e:
        run_log.append({"ticker": ticker, "status": "error", "error": str(e)})
        print(f"{ticker}: ERROR - {e}")

if all_results:
    results = pd.concat(all_results, ignore_index=True)
    results = results.sort_values(["decision", "bch_score", "prob_20_proxy_%"], ascending=[True, False, False])
else:
    results = pd.DataFrame()

print("Total rows found:", len(results))
try:
    display(results.head(25))
except NameError:
    print(results.head(25).to_string(index=False) if not results.empty else "No results to display")

In [ ]:
# =========================
# BCH Top Opportunities Dashboard Table
# =========================

DISPLAY_COLS = [
    "ticker", "spot", "expiration", "dte", "long_call", "short_call",
    "debit_mid", "max_roi_%", "prob_20_proxy_%", "prob_36_proxy_%",
    "breakout_score", "iv_score", "gamma_score", "liquidity_score",
    "bch_score", "decision"
]

if results.empty:
    ranked = pd.DataFrame(columns=DISPLAY_COLS)
    print("No results found. Check data access, ticker options availability, or loosen filters.")
else:
    ranked = results.sort_values("bch_score", ascending=False).head(20).copy()
    existing_display_cols = [c for c in DISPLAY_COLS if c in ranked.columns]
    try:
        display(ranked[existing_display_cols])
    except NameError:
        print(ranked[existing_display_cols].to_string(index=False))

In [ ]:
# =========================
# Visuals - Saved for GitHub Artifact
# =========================

chart_files = []

if not results.empty:
    top = results.sort_values("bch_score", ascending=False).head(15).copy()

    plt.figure(figsize=(12, 6))
    labels = top["ticker"] + " " + top["long_call"].astype(str) + "/" + top["short_call"].astype(str)
    plt.bar(labels, top["bch_score"])
    plt.xticks(rotation=75, ha="right")
    plt.ylabel("BCH Score")
    plt.title("BCH Top Bull Call Spread Opportunity Scores")
    plt.tight_layout()
    score_chart = OUTPUT_DIR / "BCH_Alpha_Hunter_Score_Chart.png"
    plt.savefig(score_chart, dpi=160, bbox_inches="tight")
    chart_files.append(str(score_chart.name))
    plt.show()

    plt.figure(figsize=(10, 6))
    plt.scatter(top["prob_20_proxy_%"], top["max_roi_%"], s=80)
    for _, row in top.iterrows():
        plt.annotate(row["ticker"], (row["prob_20_proxy_%"], row["max_roi_%"]))
    plt.xlabel("Probability Proxy of +20% ROI")
    plt.ylabel("Max ROI %")
    plt.title("Probability vs Max ROI")
    plt.grid(True)
    plt.tight_layout()
    prob_chart = OUTPUT_DIR / "BCH_Alpha_Hunter_Probability_vs_ROI.png"
    plt.savefig(prob_chart, dpi=160, bbox_inches="tight")
    chart_files.append(str(prob_chart.name))
    plt.show()
else:
    print("No charts created because no results were generated.")

In [ ]:
# =========================
# BCH Approval Sheet Generator
# =========================

def bch_approval_sheet(row):
    return f'''
BCH ALPHA PROBLEM ENGINE — OPPORTUNITY REPORT
======================================================================
Underlying: {row["ticker"]}
Spot: ${row["spot"]}
Expiration: {row["expiration"]}
DTE: {row["dte"]}

Structure: Bull Call Spread
Buy Call: ${row["long_call"]}
Sell Call: ${row["short_call"]}
Spread Width: ${row["width"]}
Estimated Mid Debit: ${row["debit_mid"]}

Max Profit Per Spread: ${row["max_profit"]}
Max ROI: {row["max_roi_%"]}%

Exit 1 Target: +20%
Target Spread Value: ${row["target_20_value"]}
Underlying Needed Proxy: ${row["underlying_needed_20_proxy"]}
Probability Proxy: {row["prob_20_proxy_%"]}%

Exit 2 Target: +36%
Target Spread Value: ${row["target_36_value"]}
Underlying Needed Proxy: ${row["underlying_needed_36_proxy"]}
Probability Proxy: {row["prob_36_proxy_%"]}%

Signal Scores:
- Breakout Continuation: {row["breakout_score"]}/100
- IV Mispricing: {row["iv_score"]}/100
- Gamma Acceleration: {row["gamma_score"]}/100
- Liquidity: {row["liquidity_score"]}/100
- Earnings Drift: {row["earnings_score"]}/100

BCH Composite Score: {row["bch_score"]}/100
Decision: {row["decision"]}

Evidence:
- Breakout: {row["breakout_reason"]}
- IV: {row["iv_reason"]}
- Gamma: {row["gamma_reason"]}
- Earnings: {row["earnings_reason"]}

Operator Note:
This is a measurable alpha problem output, not a manual guess. Verify live option chain, bid/ask, greeks, event risk, and broker execution quality before trade approval.
======================================================================
'''


In [ ]:
# =========================
# BCH Approval Sheet Output
# =========================

best = None
approval_text = "No BCH opportunity report generated because no results were produced."

if not results.empty:
    best = results.sort_values("bch_score", ascending=False).iloc[0]
    approval_text = bch_approval_sheet(best)

print(approval_text)

In [ ]:
# =========================
# GitHub-Ready Exports / Dashboard Data
# Always creates stable filenames for artifacts.
# =========================

# Stable artifact files
RESULTS_CSV = OUTPUT_DIR / "BCH_Alpha_Hunter_Results.csv"
TOP20_CSV = OUTPUT_DIR / "BCH_Alpha_Hunter_Top20.csv"
DASHBOARD_CSV = OUTPUT_DIR / "BCH_Alpha_Hunter_Dashboard.csv"
RUN_LOG_CSV = OUTPUT_DIR / "BCH_Alpha_Hunter_Run_Log.csv"
REPORT_MD = OUTPUT_DIR / "BCH_Alpha_Hunter_Report.md"
METADATA_JSON = OUTPUT_DIR / "BCH_Alpha_Hunter_Metadata.json"
RUN_TIMESTAMP_TXT = OUTPUT_DIR / "BCH_Alpha_Hunter_Run_Timestamp.txt"

# Save results even when empty, so GitHub artifact upload never fails.
results.to_csv(RESULTS_CSV, index=False)
ranked.to_csv(TOP20_CSV, index=False)
pd.DataFrame(run_log).to_csv(RUN_LOG_CSV, index=False)

qualified_count = int((results["decision"] == "BCH QUALIFIED").sum()) if not results.empty and "decision" in results else 0
research_count = int((results["decision"] == "RESEARCH").sum()) if not results.empty and "decision" in results else 0
watch_count = int((results["decision"] == "WATCH").sum()) if not results.empty and "decision" in results else 0

best_payload = {}
if best is not None:
    for k, v in best.to_dict().items():
        try:
            if pd.isna(v):
                best_payload[k] = None
            elif isinstance(v, (np.integer, np.floating)):
                best_payload[k] = float(v)
            else:
                best_payload[k] = v
        except Exception:
            best_payload[k] = str(v)

metadata = {
    "system": "BCH Alpha Hunter Engine",
    "run_timestamp_utc": RUN_TS,
    "run_id": RUN_ID,
    "github_actions_mode": IS_GITHUB_ACTIONS,
    "universe": BCH_UNIVERSE,
    "rows_found": int(len(results)),
    "qualified_count": qualified_count,
    "research_count": research_count,
    "watch_count": watch_count,
    "best_opportunity": best_payload,
    "output_files": [
        RESULTS_CSV.name,
        TOP20_CSV.name,
        DASHBOARD_CSV.name,
        RUN_LOG_CSV.name,
        REPORT_MD.name,
        METADATA_JSON.name,
        RUN_TIMESTAMP_TXT.name,
        *chart_files,
    ],
    "run_log": run_log,
}

summary_rows = [
    {"metric": "run_timestamp_utc", "value": RUN_TS},
    {"metric": "universe_count", "value": len(BCH_UNIVERSE)},
    {"metric": "rows_found", "value": len(results)},
    {"metric": "qualified_count", "value": qualified_count},
    {"metric": "research_count", "value": research_count},
    {"metric": "watch_count", "value": watch_count},
    {"metric": "top_ticker", "value": best_payload.get("ticker") if best_payload else None},
    {"metric": "top_bch_score", "value": best_payload.get("bch_score") if best_payload else None},
    {"metric": "top_decision", "value": best_payload.get("decision") if best_payload else None},
]
pd.DataFrame(summary_rows).to_csv(DASHBOARD_CSV, index=False)

with open(METADATA_JSON, "w") as f:
    json.dump(metadata, f, indent=2, default=str)

with open(RUN_TIMESTAMP_TXT, "w") as f:
    f.write(f"BCH Alpha Hunter run completed at {RUN_TS}\n")
    f.write(f"Rows found: {len(results)}\n")
    f.write(f"Qualified: {qualified_count}\n")
    f.write(f"Research: {research_count}\n")
    f.write(f"Watch: {watch_count}\n")

report = f"""# BCH Alpha Hunter Daily Report

**Run timestamp UTC:** {RUN_TS}  
**Universe:** {', '.join(BCH_UNIVERSE)}  
**Rows found:** {len(results)}  
**Qualified:** {qualified_count}  
**Research:** {research_count}  
**Watch:** {watch_count}  

## Top Opportunity

```text
{approval_text}
```

## Artifact Files

- {RESULTS_CSV.name}
- {TOP20_CSV.name}
- {DASHBOARD_CSV.name}
- {RUN_LOG_CSV.name}
- {METADATA_JSON.name}
- {RUN_TIMESTAMP_TXT.name}
"""

with open(REPORT_MD, "w") as f:
    f.write(report)

print("BCH Alpha Hunter Engine completed.")
print("Exported files:")
for f in metadata["output_files"]:
    print("-", f)

## GitHub Automation Process

1. Upload this notebook to the root of your GitHub repository.
2. Add the provided workflow file at `.github/workflows/alpha_hunter_daily.yml`.
3. Run the workflow manually once from the GitHub **Actions** tab.
4. Download the artifact named `BCH-Alpha-Hunter-Daily-Reports`.
5. Tomorrow, verify the scheduled run appears with event type `schedule`.

The workflow uses the stable report filenames created by this notebook.